# Block 07: Multi-head attention, in NumPy and then PyTorch

**Paper anchor:** §3.2.2, Figure 2 (right), Table 3 rows (A)
**Builds on:** block 06; prerequisite P6
**Time:** about 6 hours, in short sessions
**You will write:** `split_heads`, `merge_heads`, `multi_head_attention` (NumPy); `MultiHeadAttention` (PyTorch)
**Done when:** all four checkpoints pass

> **How this notebook works.** Claude teaches each concept in chat, one step at a time (type `next`).
> You write every line of solution code, in the cells marked **YOUR CODE**. Checkpoint cells test behavior only
> and never contain a solution (please don't open `checks/` until you pass). Stuck or wrong? Paste your code and
> the output into chat: we find the misconception together. Log every mistake in the table at the bottom.

In [ ]:
# provided: repo paths (works from notebooks/ locally and from the repo root on Colab)
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
np.set_printoptions(precision=4, suppress=True)
from checks import b07
from minitransformer.numpy_ops import softmax, scaled_dot_product_attention, causal_mask  # graduated

## Big picture
One attention head computes one weighted average per position. Multi-head attention runs h smaller attentions in
parallel on different learned projections and concatenates them (§3.2.2):

$$\mathrm{head}_i = \mathrm{Attention}(XW_i^Q, XW_i^K, XW_i^V), \qquad \mathrm{MultiHead} = \mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_h)W^O$$

Paper sizes: $d_{model} = 512$, $h = 8$, $d_k = d_v = d_{model}/h = 64$.

| step | shape |
|---|---|
| X | (B, T, d_model) |
| X @ Wq (all heads packed) | (B, T, h·d_k) |
| split_heads | (B, h, T, d_k) |
| attention per head | (B, h, T, d_k) |
| merge_heads | (B, T, h·d_k) |
| @ Wo | (B, T, d_model) |

**Notation traps.** $d_k$ is per head; $d_{model}$ is total. The packed weight's columns $i \cdot d_k$ to $(i+1) d_k$ belong to head $i$.

## Before you code
Answer in the cell below, in your own words. Guessing is fine; we check in chat.

1. With d_model = 512 and h = 8, what is d_k, and how many parameters do W^Q, W^K, W^V, W^O hold in total?
2. Why does reshaping (B, T, d_model) straight to (B, h, T, d_k) give wrong heads?
3. Table 3 (A) varies h while keeping computation constant. Why is the cost roughly constant?

*My answers:*

1. 
2. 
3. 

## Part A: NumPy

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

def split_heads(X, h):
    """(B, T, d_model) -> (B, h, T, d_k) with d_k = d_model // h.

    Head i owns columns i*d_k : (i+1)*d_k of each position's vector.
    """
    raise NotImplementedError("Block 07: your turn")

def merge_heads(X):
    """(B, h, T, d_k) -> (B, T, h * d_k). Exact inverse of split_heads."""
    raise NotImplementedError("Block 07: your turn")

def multi_head_attention(X, Wq, Wk, Wv, Wo, h, mask=None):
    """Multi-head self-attention, §3.2.2, with packed weights and no biases.

    X:  (B, T, d_model)
    Wq, Wk, Wv: (d_model, d_model). Columns i*d_k : (i+1)*d_k are head i's W_i^Q (resp. K, V).
    Wo: (d_model, d_model), the paper's W^O (rows i*d_k : (i+1)*d_k act on head i's output).
    h:  number of heads; d_model must be divisible by h.
    mask: None, or bool broadcastable to (B, h, T, T); True = allowed. A causal mask is (T, T);
          a key-padding mask is (B, 1, 1, T).
    returns: (out (B, T, d_model), weights (B, h, T, T))
    Biases: the paper gives only matrices (§3.2.2); this repo's NumPy version has none (assumption).
    """
    raise NotImplementedError("Block 07: your turn")

In [ ]:
b07.check_split_merge(split_heads, merge_heads)
b07.check_mha_numpy(multi_head_attention, scaled_dot_product_attention, causal_mask)

## Part B0: PyTorch warm-up
If PyTorch is new, we do a short primer in chat first: tensors, autograd, `nn.Module`, and the transpose inside `nn.Linear`.
Exercise: rebuild block 03's cross-entropy gradient with `torch.autograd` and compare it with your hand-derived `dlogits`.

In [ ]:
# YOUR CODE: autograd vs your NumPy gradient

## Part B: PyTorch module
`context` makes the same module usable for encoder-decoder attention in block 15 (§3.2.3, first bullet).
Dropout on attention weights is an assumption: §5.4 does not list it.

In [ ]:
import torch, torch.nn as nn

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

class MultiHeadAttention(nn.Module):
    """Multi-head attention, §3.2.2 and Fig. 2 (right).

    __init__(d_model, n_heads, dropout=0.0, bias=False)
        Attributes: W_q, W_k, W_v, W_o, each nn.Linear(d_model, d_model, bias=bias).
        d_k = d_model // n_heads. `dropout` is applied to the attention weights
        (an assumption: §5.4 does not list it; tensor2tensor does it).

    forward(x, context=None, mask=None) -> (B, T_q, d_model)
        x:       (B, T_q, d_model), source of the queries
        context: (B, T_k, d_model) source of keys and values; None means self-attention (context = x)
        mask:    None, or bool tensor broadcastable to (B, n_heads, T_q, T_k); True = allowed.
                 A causal mask has shape (T_q, T_k); a key-padding mask has shape (B, 1, 1, T_k).
    Trap: nn.Linear computes x @ weight.T, so the paper's W^Q is W_q.weight.T.
    """

    def __init__(self, d_model, n_heads, dropout=0.0, bias=False):
        super().__init__()
        raise NotImplementedError("Block 07: your turn")

    def forward(self, x, context=None, mask=None):
        raise NotImplementedError("Block 07: your turn")

In [ ]:
from minitransformer.numpy_ops import multi_head_attention as mha_numpy  # graduate Part A first
b07.check_torch_mha(MultiHeadAttention, mha_numpy)

## Explain it back
Without notes. Then paste your answers into chat.

1. Why several small heads instead of one big one? Use §3.2.2's own argument and Table 3 (A).
2. Walk the shapes from X to the output without looking.
3. Why is the paper's W^Q equal to `W_q.weight.T`?

*My explanation:*

1. 
2. 
3. 

## Mistakes log
Every wrong attempt is data. Copy finished rows into `journal/mistakes.md`.

| # | What I wrote | Symptom | The misconception | The fix |
|---|---|---|---|---|
| 1 | | | | |

## Graduate
NumPy functions into `numpy_ops.py`; the class into `src/minitransformer/attention.py`. Run `pytest -q`. Commit `block 07: multi-head attention`. Mark P6 done.